# Downloading the Osborne Mine, Australia,  airborne magnetic survey 

Accessed from https://ecat.ga.gov.au/geonetwork/srv/eng/catalog.search#/metadata/142419

Inspired by: https://github.com/fatiando-data/osborne-magnetic/blob/main/prepare.ipynb

In [ ]:
import numpy as np
import pandas as pd
import pooch
import pygmt
import pyproj
import verde as vd
import xarray as xr

import airbornegeo

## load data

In [ ]:
fname = pooch.retrieve(
    url="https://thredds.nci.org.au/thredds/fileServer/iv65/Geoscience_Australia_Geophysics_Reference_Data_Collection/airborne_geophysics/QLD/line/P1029/P1029-line-magnetic-AWAGS_MAG_2010.nc",
    known_hash="sha256:119b472da05365f0df4e9dc0b2d4b0e5c213705acb4f79efcaa07e1aeeb7242c",
    progressbar=True,
    path=f"{pooch.os_cache('airbornegeo')}",
)
data_ds = xr.open_dataset(fname)
data_ds

In [ ]:
data_df = pd.DataFrame(
    {
        "longitude_gda": data_ds.longitude.data,
        "latitude_gda": data_ds.latitude.data,
        "terrain_clearance_m": data_ds.altitude.data.astype(np.float32),
        "total_field_anomaly_awags_levelled": data_ds.mag_awagsLevelled.data.astype(
            np.float32
        ),
        "total_field_anomaly_microlevelled": data_ds.mag_microLevelled.data.astype(
            np.float32
        ),
        "total_field_anomaly_levelled": data_ds.mag_tieLevelled.data.astype(np.float32),
        "flight_line": data_ds.line_index.data.astype(np.uint16),
    }
)
data_df

## Reproject

In [ ]:
data_df["longitude"], data_df["latitude"] = airbornegeo.reproject(
    data_df.longitude_gda,
    data_df.latitude_gda,
    input_crs="epsg:4283",
    output_crs="epsg:4326",
)
region = vd.get_region((data_df.longitude, data_df.latitude))
data_df

## Correct heights

In [ ]:
srtm = vd.grid_to_table(
    pygmt.grdcut("@earth_relief_01s_g", region=vd.pad_region(region, 2 / 60))
)
srtm

In [ ]:
projection = pyproj.Proj(proj="merc", lat_ts=data_df.latitude.mean())

nearest = vd.KNeighbors()
nearest.fit(projection(srtm.lon.values, srtm.lat.values), srtm.z)
topography = nearest.predict(
    projection(data_df.longitude.values, data_df.latitude.values)
)

data_df = data_df.assign(
    height_orthometric_m=topography + data_df.terrain_clearance_m,
)
data_df

In [ ]:
data = data_df[::100]

fig = pygmt.Figure()
with fig.subplot(
    nrows=1,
    ncols=2,
    figsize=("30c", "20c"),
    sharey="l",  # shared y-axis on the left side
    frame="WSrt",
):
    with fig.set_panel(0):
        fig.basemap(projection="M?", region=region, frame="af")
        scale = 1500
        pygmt.makecpt(cmap="polar+h", series=[-scale, scale], background=True)
        fig.plot(
            x=data.longitude,
            y=data.latitude,
            fill=data.total_field_anomaly_awags_levelled,
            style="c0.02c",
            cmap=True,
        )
        fig.colorbar(
            frame="af+ltotal field magnetic anomaly [nT]",
            position="JBC+h+o0/1c+e",
        )
    with fig.set_panel(1):
        fig.basemap(projection="M?", region=region, frame="af")
        pygmt.makecpt(
            cmap="viridis",
            series=[data.height_orthometric_m.min(), data.height_orthometric_m.max()],
        )
        fig.plot(
            x=data.longitude,
            y=data.latitude,
            fill=data.height_orthometric_m,
            style="c0.02c",
            cmap=True,
        )
        fig.colorbar(
            frame="af+lobservation height [m]",
            position="JBC+h+o0/1c",
        )
fig.show(width=800)

## Subset a region

In [ ]:
# West, East, South, North
region = (140 + 30 / 60, 140 + 50 / 60, -22 - 10 / 60, -21 - 45 / 60)
selection = vd.inside((data_df.longitude_gda, data_df.latitude_gda), region)
data = data_df[selection].reset_index(drop=True).copy()
data

In [ ]:
airbornegeo.plotly_points(
    data_df[::10],  # plot every 10th point
    color_col="line",
    hover_cols=["line_name"],
    robust=False,
    size=3,
)

In [ ]:
# drop a few lines to clean up the survey
data_df = data_df[
    ~data_df.line_name.isin(
        [
            "BYRD1_69.0",
            "BYRD2_69.0",
            "BYRD3_69.0",
            "TAM1_4.0",
            "CTAM1_5.0",
            "SP10030_65.0",
            "F10291_37.0",
            "V10250_41.0",
        ]
    )
]

In [ ]:
# convert supplied line names and flights into integers
data_df["line"] = airbornegeo.unique_line_id(data_df, line_col_name="line_name")

# drop unneeded columns
data_df = data_df.drop(columns=["Flight_ID", "Line_no"])

# drop rows with all NaNs
data_df = data_df.dropna(how="all")

data_df

In [ ]:
airbornegeo.plotly_points(
    data_df[::10],
    color_col="line",
    hover_cols=["line_name"],
    robust=False,
    size=3,
)

In [ ]:
# calculate the unixtime from the date and time columns
data_df["Date"] = pd.to_datetime(data_df["Date"])
data_df["Time"] = pd.to_timedelta(data_df["Time"])
data_df["unixtime"] = data_df["Date"] + data_df["Time"]
data_df = data_df.dropna(subset=["unixtime"])
data_df["unixtime"] = data_df["unixtime"].apply(lambda x: x.timestamp())
data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)
data_df.head()

In [ ]:
airbornegeo.plotly_points(
    data_df[::10],
    color_col="unixtime",
    hover_cols=["line"],
    robust=False,
    size=3,
)

In [ ]:
data_df.to_csv("../data/AGAP_magnetic_survey.csv", index=False)